In [1]:
from pyspark.sql import SparkSession


In [2]:
spark = SparkSession.builder.appName("NYCTaxi").getOrCreate()

In [3]:
df = spark.read.parquet("/content/drive/MyDrive/PySparkprojectData/yellow_tripdata_2023-01.parquet")

In [4]:
import requests

def download_taxi_data(year, month, save_path):
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet"
    filename = f"{save_path}/yellow_tripdata_{year}-{month:02d}.parquet"

    response = requests.get(url, stream=True)
    response.raise_for_status()

    with open(filename, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Downloaded: {filename}")
    return filename

In [9]:
download_taxi_data(2023, 1, '/content/drive/MyDrive/PySparkprojectData')

Downloaded: /content/drive/MyDrive/PySparkprojectData/yellow_tripdata_2023-01.parquet


'/content/drive/MyDrive/PySparkprojectData/yellow_tripdata_2023-01.parquet'

In [16]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2023-01-01 00:32:10|  2023-01-01 00:40:36|            1.0|         0.97|       1.0|                 N|         161|         141|           2|        9.3|  1.0|    0.5|       0.

In [17]:
df.count()

3066766

In [33]:
import os

def get_data_from_to(FromMonth, FromYear, ToMonth, ToYear, FolderPath):
    if not (1 <= FromMonth <= 12 and 1 <= ToMonth <= 12):
      raise ValueError("Month must be between 1 and 12.")

    if not (2009 <= FromYear <= 2026 and 2009 <= ToYear <= 2026):
      raise ValueError("Year must be between 2009 and 2026.")

    from_date = (FromYear, FromMonth)
    to_date   = (ToYear, ToMonth)

    if from_date >= to_date:
        raise ValueError(
            f"'From' date ({FromMonth}/{FromYear}) "
            f"must be before 'To' date ({ToMonth}/{ToYear}).")

    all_files = os.listdir(FolderPath)

    current_month = FromMonth
    last_month = 12

    for year in range(FromYear, ToYear + 1):

      if year == ToYear:
        last_month = ToMonth

      for month in range(current_month, last_month + 1):
        url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet"
        filename = f"{FolderPath}/yellow_tripdata_{year}-{month:02d}.parquet"

        if filename[len(FolderPath)+1:] not in all_files:
          print(f"fetching {filename}")
          download_taxi_data(filename, url, FolderPath)

      current_month = 1

def download_taxi_data(filename, url, save_path):

  response = requests.get(url, stream=True)
  response.raise_for_status()

  with open(filename, "wb") as f:
      for chunk in response.iter_content(chunk_size=8192):
          f.write(chunk)
  print(f"Downloaded: {filename}")

In [31]:
get_data_from_to(12, 2022, 1, 2023, '/content/drive/MyDrive/PySparkprojectData')

fetching /content/drive/MyDrive/PySparkprojectData/yellow_tripdata_2022-12.parquet
Downloaded: /content/drive/MyDrive/PySparkprojectData/yellow_tripdata_2022-12.parquet
fetching /content/drive/MyDrive/PySparkprojectData/yellow_tripdata_2023-01.parquet
Downloaded: /content/drive/MyDrive/PySparkprojectData/yellow_tripdata_2023-01.parquet


In [32]:
get_data_from_to(12, 2022, 2, 2023, '/content/drive/MyDrive/PySparkprojectData')

fetching /content/drive/MyDrive/PySparkprojectData/yellow_tripdata_2023-02.parquet
Downloaded: /content/drive/MyDrive/PySparkprojectData/yellow_tripdata_2023-02.parquet
